# Bird Migration — Data Preprocessing & Feature Engineering
**Author:** Jayant Pandey  
**Project:** Bird Migration Trajectory Prediction using GRU  
**Dataset:** GPS tracking data — 3 White Storks (Eric, Nico, Sanne), Aug 2013 – Apr 2014  
**Scope:** Data ingestion, cleaning, feature engineering, climate data integration (Open-Meteo), MinMaxScaler preprocessing, GRU sequence creation, inverse transform post-processing

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import requests
import time
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler

print('All imports successful')

All imports successful


## 2. Data Ingestion

In [2]:
df = pd.read_csv('bird_migration.csv')

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print()
df.head(3)

Shape: (61920, 9)
Columns: ['Unnamed: 0', 'altitude', 'date_time', 'device_info_serial', 'direction', 'latitude', 'longitude', 'speed_2d', 'bird_name']



,Unnamed: 0,altitude,date_time,device_info_serial,direction,latitude,longitude,speed_2d,bird_name
0,0,71,2013-08-15 00:18:08+00,851,-150.469753,49.41986,2.120733,0.150000,Eric
1,1,68,2013-08-15 00:48:07+00,851,-136.151141,49.41988,2.120746,2.438360,Eric
2,2,68,2013-08-15 01:17:58+00,851,160.797477,49.42031,2.120885,0.596657,Eric


In [3]:
print('Data types:')
print(df.dtypes)
print()
print('Null counts:')
print(df.isnull().sum())

Data types:
Unnamed: 0              int64
altitude                int64
date_time              object
device_info_serial      int64
direction             float64
latitude              float64
longitude             float64
speed_2d              float64
bird_name              object
dtype: object

Null counts:
Unnamed: 0              0
altitude                0
date_time               0
device_info_serial      0
direction             443
latitude                0
longitude               0
speed_2d              443
bird_name               0
dtype: int64


In [4]:
print('Birds:', df['bird_name'].unique())
print()
print('Records per bird:')
print(df['bird_name'].value_counts())
print()
print('Date range:', df['date_time'].min(), 'to', df['date_time'].max())

Birds: ['Eric' 'Nico' 'Sanne']

Records per bird:
bird_name
Nico     21121
Sanne    21004
Eric     19795
Name: count, dtype: int64

Date range: 2013-08-15 00:01:08+00 to 2014-04-30 23:59:34+00


## 3. Data Cleaning

In [5]:
df = df.drop(columns=['Unnamed: 0'])
print('Dropped Unnamed:0. Shape now:', df.shape)

Dropped Unnamed:0. Shape now: (61920, 8)


In [6]:
df['date_time'] = pd.to_datetime(df['date_time'], utc=True)
print('date_time dtype:', df['date_time'].dtype)
print('Sample:', df['date_time'].iloc[0])

date_time dtype: datetime64[ns, UTC]
Sample: 2013-08-15 00:18:08+00:00


In [8]:
df = df.sort_values(['bird_name', 'date_time']).reset_index(drop=True)
print('Sorted. Sample:')
df[['bird_name', 'date_time', 'latitude', 'longitude']].head(4)

Sorted. Sample:


,bird_name,date_time,latitude,longitude
0,Eric,2013-08-15 00:18:08+00:00,49.419860,2.120733
1,Eric,2013-08-15 00:48:07+00:00,49.419880,2.120746
2,Eric,2013-08-15 01:17:58+00:00,49.420310,2.120885
3,Eric,2013-08-15 01:47:51+00:00,49.420359,2.120859


In [7]:
df['direction'] = df.groupby('bird_name')['direction'].transform(lambda x: x.ffill())
df['speed_2d']  = df.groupby('bird_name')['speed_2d'].transform(lambda x: x.ffill())

print('Nulls after forward fill:')
print(df.isnull().sum())

Nulls after forward fill:
altitude              0
date_time             0
device_info_serial    0
direction             0
latitude              0
longitude             0
speed_2d              0
bird_name             0
dtype: int64


In [8]:
df = df.dropna().reset_index(drop=True)
print('Final shape after cleaning:', df.shape)

Final shape after cleaning: (61920, 8)


## 4. Feature Engineering

Three temporal features created for GRU input:
- `unix_timestamp` — exact chronological ordering between GPS points
- `day_of_year` — seasonal migration pattern (1–365)
- `hour` — intra-day movement behaviour (0–23)

In [9]:
df['unix_timestamp'] = df['date_time'].astype(np.int64) // 10**9

df['day_of_year'] = df['date_time'].dt.dayofyear

df['hour'] = df['date_time'].dt.hour

print('New columns added:')
df[['date_time', 'unix_timestamp', 'day_of_year', 'hour']].head(4)

New columns added:


,date_time,unix_timestamp,day_of_year,hour
0,2013-08-15 00:18:08+00:00,1376525888,227,0
1,2013-08-15 00:48:07+00:00,1376527687,227,0
2,2013-08-15 01:17:58+00:00,1376529478,227,1
3,2013-08-15 01:47:51+00:00,1376531271,227,1


In [12]:
GRU_FEATURES   = ['latitude', 'longitude', 'unix_timestamp', 'day_of_year', 'hour']
TARGET_INDICES = [0, 1]
SEQ_LENGTH     = 10

print('GRU features:')
print(df[GRU_FEATURES].isnull().sum())
print()
df[GRU_FEATURES].head(3)

GRU features:
latitude          0
longitude         0
unix_timestamp    0
day_of_year       0
hour              0
dtype: int64



,latitude,longitude,unix_timestamp,day_of_year,hour
0,49.41986,2.120733,1376525888,227,0
1,49.41988,2.120746,1376527687,227,0
2,49.42031,2.120885,1376529478,227,1


## 5. Climate Data Integration (Open-Meteo Archive API)

Historical weather fetched per GPS coordinate: `temperature_2m` (°C), `wind_speed_10m` (km/h), `precipitation` (mm).  
Parallel fetch using `ThreadPoolExecutor`. Checkpoint saving every 500 rows for fault tolerance.  
> **Note:** 10,388 GPS points over West Africa (lat 12–16°N, lon 16–17°W) had no hourly data in Open-Meteo archive. Forward fill applied within each bird group.

In [10]:
sample = df.iloc[0]
lat    = round(float(sample['latitude']), 4)
lon    = round(float(sample['longitude']), 4)
date   = sample['date_time'].strftime('%Y-%m-%d')
hour   = int(sample['hour'])

url = (
    f"https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={date}&end_date={date}"
    f"&hourly=temperature_2m,wind_speed_10m,precipitation"
    f"&timezone=UTC"
)

response = requests.get(url)
data     = response.json()

print('Status:', response.status_code)
print('Hourly keys:', list(data.get('hourly', {}).keys()))
print('Temperature at hour', hour, ':', data['hourly']['temperature_2m'][hour])

Status: 200
Hourly keys: ['time', 'temperature_2m', 'wind_speed_10m', 'precipitation']
Temperature at hour 0 : 14.3


In [11]:
def fetch_climate(lat, lon, date_str, hour):
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={round(lat,4)}&longitude={round(lon,4)}"
        f"&start_date={date_str}&end_date={date_str}"
        f"&hourly=temperature_2m,wind_speed_10m,precipitation"
        f"&timezone=UTC"
    )
    try:
        r      = requests.get(url, timeout=10)
        d      = r.json()
        hourly = d.get('hourly', {})
        temp   = hourly.get('temperature_2m',  [np.nan]*24)[hour]
        wind   = hourly.get('wind_speed_10m',  [np.nan]*24)[hour]
        precip = hourly.get('precipitation',   [np.nan]*24)[hour]
        return temp, wind, precip
    except:
        return np.nan, np.nan, np.nan

In [12]:
test_df = df.head(50).copy()
temps, winds, precips = [], [], []

for i, row in test_df.iterrows():
    t, w, p = fetch_climate(
        row['latitude'], row['longitude'],
        row['date_time'].strftime('%Y-%m-%d'),
        int(row['hour'])
    )
    temps.append(t); winds.append(w); precips.append(p)
    if i % 10 == 0:
        print(f'Row {i} done — temp: {t}, wind: {w}, precip: {p}')
    time.sleep(0.1)

test_df['temperature']   = temps
test_df['wind_speed']    = winds
test_df['precipitation'] = precips

print('\nNull check on test batch:')
print(test_df[['temperature','wind_speed','precipitation']].isnull().sum())

Row 0 done — temp: 14.3, wind: 5.4, precip: 0.0
Row 10 done — temp: 15.0, wind: 2.4, precip: 0.0
Row 20 done — temp: 15.4, wind: 6.5, precip: 0.0
Row 30 done — temp: 20.3, wind: 9.8, precip: 0.0
Row 40 done — temp: 23.0, wind: 9.6, precip: 0.0

Null check on test batch:
temperature      0
wind_speed       0
precipitation    0
dtype: int64


In [6]:
import os
import pandas as pd
import numpy as np
import requests
import time

df = pd.read_csv('bird_migration.csv')
df = df.drop(columns=['Unnamed: 0'])
df['date_time'] = pd.to_datetime(df['date_time'], utc=True)
df = df.sort_values(['bird_name', 'date_time']).reset_index(drop=True)
df['direction'] = df.groupby('bird_name')['direction'].transform(lambda x: x.ffill())
df['speed_2d']  = df.groupby('bird_name')['speed_2d'].transform(lambda x: x.ffill())
df = df.dropna().reset_index(drop=True)
df['unix_timestamp'] = df['date_time'].astype(np.int64) // 10**9
df['day_of_year']    = df['date_time'].dt.dayofyear
df['hour']           = df['date_time'].dt.hour
print('df ready. Shape:', df.shape)

def fetch_climate(lat, lon, date_str, hour):
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={round(lat,4)}&longitude={round(lon,4)}"
        f"&start_date={date_str}&end_date={date_str}"
        f"&hourly=temperature_2m,wind_speed_10m,precipitation"
        f"&timezone=UTC"
    )
    for attempt in range(3):
        try:
            r      = requests.get(url, timeout=15)
            d      = r.json()
            hourly = d.get('hourly', {})
            temp   = hourly.get('temperature_2m',  [np.nan]*24)[hour]
            wind   = hourly.get('wind_speed_10m',  [np.nan]*24)[hour]
            precip = hourly.get('precipitation',   [np.nan]*24)[hour]
            if temp is not None:
                return temp, wind, precip
        except:
            time.sleep(2)
    return np.nan, np.nan, np.nan

print('Fetch function defined.')

checkpoint = pd.read_csv('climate_checkpoint.csv')
already_done = len(checkpoint)
print(f'Resuming from row {already_done} of {len(df)}')

remaining_df = df.iloc[already_done:].copy()

all_temps, all_winds, all_precips = [], [], []

for i, row in remaining_df.iterrows():
    t, w, p = fetch_climate(
        row['latitude'], row['longitude'],
        row['date_time'].strftime('%Y-%m-%d'),
        int(row['hour'])
    )
    all_temps.append(t)
    all_winds.append(w)
    all_precips.append(p)

    if i % 1500 == 0:
        print(f'Progress: {i}/{len(df)} rows done')
        temp_remaining = remaining_df.iloc[:len(all_temps)].copy()
        temp_remaining['temperature']   = all_temps
        temp_remaining['wind_speed']    = all_winds
        temp_remaining['precipitation'] = all_precips
        updated = pd.concat([checkpoint, temp_remaining], ignore_index=True)
        updated.to_csv('climate_checkpoint.csv', index=False)

    time.sleep(0.1)

remaining_df['temperature']   = all_temps
remaining_df['wind_speed']    = all_winds
remaining_df['precipitation'] = all_precips

final_df = pd.concat([checkpoint, remaining_df], ignore_index=True)
final_df.to_csv('bird_migration_with_climate.csv', index=False)

print('Done.')
print('Final shape:', final_df.shape)
print('Null check:')
print(final_df[['temperature','wind_speed','precipitation']].isnull().sum())

df ready. Shape: (61920, 11)
Fetch function defined.
Resuming from row 25501 of 61920
Progress: 27000/61920 rows done
Progress: 28500/61920 rows done
Progress: 30000/61920 rows done
Progress: 31500/61920 rows done
Progress: 33000/61920 rows done
Progress: 34500/61920 rows done
Progress: 36000/61920 rows done
Progress: 37500/61920 rows done
Progress: 39000/61920 rows done
Progress: 40500/61920 rows done
Progress: 42000/61920 rows done
Progress: 43500/61920 rows done
Progress: 45000/61920 rows done
Progress: 46500/61920 rows done
Progress: 48000/61920 rows done
Progress: 49500/61920 rows done
Progress: 51000/61920 rows done
Progress: 52500/61920 rows done
Progress: 54000/61920 rows done
Progress: 55500/61920 rows done
Progress: 57000/61920 rows done
Progress: 58500/61920 rows done
Progress: 60000/61920 rows done
Progress: 61500/61920 rows done
Done.
Final shape: (61920, 14)
Null check:
temperature      20342
wind_speed       20342
precipitation    20342
dtype: int64


In [7]:
import pandas as pd

checkpoint = pd.read_csv('climate_checkpoint.csv')
print('Total rows:', len(checkpoint))
print('Null check:')
print(checkpoint[['temperature','wind_speed','precipitation']].isnull().sum())

Total rows: 61501
Null check:
temperature      20342
wind_speed       20342
precipitation    20342
dtype: int64


In [1]:
import pandas as pd

checkpoint = pd.read_csv('climate_checkpoint.csv')
print('Checkpoint rows:', len(checkpoint))
print('Null check:')
print(checkpoint[['temperature','wind_speed','precipitation']].isnull().sum())

Checkpoint rows: 61501
Null check:
temperature      20341
wind_speed       20341
precipitation    20341
dtype: int64


In [4]:
import requests
import time
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed

checkpoint = pd.read_csv('climate_checkpoint.csv')

def fetch_climate_single(args):
    idx, lat, lon, date_str, hour = args
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={round(lat,4)}&longitude={round(lon,4)}"
        f"&start_date={date_str}&end_date={date_str}"
        f"&hourly=temperature_2m,wind_speed_10m,precipitation"
        f"&timezone=UTC"
    )
    for attempt in range(3):
        try:
            r      = requests.get(url, timeout=15)
            d      = r.json()
            hourly = d.get('hourly', {})
            temp   = hourly.get('temperature_2m',  [np.nan]*24)[hour]
            wind   = hourly.get('wind_speed_10m',  [np.nan]*24)[hour]
            precip = hourly.get('precipitation',   [np.nan]*24)[hour]
            if temp is not None:
                return idx, temp, wind, precip
        except:
            time.sleep(1)
    return idx, np.nan, np.nan, np.nan

null_rows = checkpoint[checkpoint['temperature'].isnull()]
tasks = [
    (idx, row['latitude'], row['longitude'],
     pd.to_datetime(row['date_time']).strftime('%Y-%m-%d'),
     int(pd.to_datetime(row['date_time']).hour))
    for idx, row in null_rows.iterrows()
]

print(f'Re-fetching {len(tasks)} null rows in parallel...')

completed = 0
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(fetch_climate_single, task): task for task in tasks}
    for future in as_completed(futures):
        idx, t, w, p = future.result()
        checkpoint.at[idx, 'temperature']   = t
        checkpoint.at[idx, 'wind_speed']    = w
        checkpoint.at[idx, 'precipitation'] = p
        completed += 1
        if completed % 1000 == 0:
            checkpoint.to_csv('climate_checkpoint.csv', index=False)
            print(f'Fixed {completed}/{len(tasks)} rows...')

checkpoint.to_csv('climate_checkpoint.csv', index=False)
print('Done. Nulls remaining:', checkpoint['temperature'].isnull().sum())

Re-fetching 10388 null rows in parallel...
Fixed 1000/10388 rows...
Fixed 2000/10388 rows...
Fixed 3000/10388 rows...
Fixed 4000/10388 rows...
Fixed 5000/10388 rows...
Fixed 6000/10388 rows...
Fixed 7000/10388 rows...
Fixed 8000/10388 rows...
Fixed 9000/10388 rows...
Fixed 10000/10388 rows...
Done. Nulls remaining: 10388


In [5]:
import requests
import pandas as pd
import numpy as np

checkpoint = pd.read_csv('climate_checkpoint.csv')
null_rows = checkpoint[checkpoint['temperature'].isnull()]

test_samples = null_rows.iloc[[0, 100, 500, 1000, 5000]]

for idx, row in test_samples.iterrows():
    lat  = round(row['latitude'], 4)
    lon  = round(row['longitude'], 4)
    date = pd.to_datetime(row['date_time']).strftime('%Y-%m-%d')
    hour = int(pd.to_datetime(row['date_time']).hour)
    
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}&longitude={lon}"
        f"&start_date={date}&end_date={date}"
        f"&hourly=temperature_2m,wind_speed_10m,precipitation"
        f"&timezone=UTC"
    )
    r = requests.get(url, timeout=15)
    d = r.json()
    temp = d.get('hourly', {}).get('temperature_2m', [None]*24)[hour]
    print(f'Row {idx}: lat={lat}, lon={lon}, date={date}, hour={hour} → temp={temp}')

Row 34491: lat=15.6858, lon=-16.7173, date=2014-02-09, hour=4 → temp=None
Row 34783: lat=15.358, lon=-16.7921, date=2014-02-12, hour=16 → temp=None
Row 36056: lat=14.6958, lon=-17.37, date=2014-02-28, hour=2 → temp=None
Row 37502: lat=13.7913, lon=-16.7497, date=2014-03-17, hour=14 → temp=None
Row 47961: lat=14.1317, lon=-16.85, date=2013-11-09, hour=6 → temp=None


In [6]:
import pandas as pd

checkpoint = pd.read_csv('climate_checkpoint.csv')
print('Nulls before fill:', checkpoint['temperature'].isnull().sum())

checkpoint['temperature']   = checkpoint.groupby('bird_name')['temperature'].transform(lambda x: x.ffill().bfill())
checkpoint['wind_speed']    = checkpoint.groupby('bird_name')['wind_speed'].transform(lambda x: x.ffill().bfill())
checkpoint['precipitation'] = checkpoint.groupby('bird_name')['precipitation'].transform(lambda x: x.ffill().bfill())

print('Nulls after fill:', checkpoint['temperature'].isnull().sum())
checkpoint.to_csv('bird_migration_with_climate.csv', index=False)
print('Saved as bird_migration_with_climate.csv. Shape:', checkpoint.shape)

Nulls before fill: 10388
Nulls after fill: 0
Saved as bird_migration_with_climate.csv. Shape: (61501, 14)


In [8]:
import pandas as pd
import numpy as np

original = pd.read_csv('bird_migration.csv')
climate  = pd.read_csv('bird_migration_with_climate.csv')

print('Original rows:', len(original))
print('Climate rows:', len(climate))
print('Missing:', len(original) - len(climate))
print()
print('Missing birds:')
orig_counts    = original['bird_name'].value_counts()
climate_counts = climate['bird_name'].value_counts()
print('Original:', orig_counts.to_dict())
print('Climate: ', climate_counts.to_dict())

Original rows: 61920
Climate rows: 61501
Missing: 419

Missing birds:
Original: {'Nico': 21121, 'Sanne': 21004, 'Eric': 19795}
Climate:  {'Nico': 21121, 'Sanne': 20585, 'Eric': 19795}


In [9]:
import pandas as pd
import numpy as np
import requests
import time

original = pd.read_csv('bird_migration.csv')
climate  = pd.read_csv('bird_migration_with_climate.csv')

original['date_time'] = pd.to_datetime(original['date_time'], utc=True)
original = original.drop(columns=['Unnamed: 0'])
original = original.sort_values(['bird_name', 'date_time']).reset_index(drop=True)
original['unix_timestamp'] = original['date_time'].astype(np.int64) // 10**9
original['day_of_year']    = original['date_time'].dt.dayofyear
original['hour']           = original['date_time'].dt.hour

missing = original[~original.index.isin(climate.index)].copy()
print('Missing rows:', len(missing))
print('Bird:', missing['bird_name'].unique())

def fetch_climate(lat, lon, date_str, hour):
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={round(lat,4)}&longitude={round(lon,4)}"
        f"&start_date={date_str}&end_date={date_str}"
        f"&hourly=temperature_2m,wind_speed_10m,precipitation"
        f"&timezone=UTC"
    )
    for attempt in range(3):
        try:
            r      = requests.get(url, timeout=15)
            d      = r.json()
            hourly = d.get('hourly', {})
            temp   = hourly.get('temperature_2m',  [np.nan]*24)[hour]
            wind   = hourly.get('wind_speed_10m',  [np.nan]*24)[hour]
            precip = hourly.get('precipitation',   [np.nan]*24)[hour]
            if temp is not None:
                return temp, wind, precip
        except:
            time.sleep(2)
    return np.nan, np.nan, np.nan

temps, winds, precips = [], [], []

for i, row in missing.iterrows():
    t, w, p = fetch_climate(
        row['latitude'], row['longitude'],
        row['date_time'].strftime('%Y-%m-%d'),
        int(row['hour'])
    )
    temps.append(t)
    winds.append(w)
    precips.append(p)
    time.sleep(0.1)

missing['temperature']   = temps
missing['wind_speed']    = winds
missing['precipitation'] = precips

missing['temperature']   = missing['temperature'].ffill().bfill()
missing['wind_speed']    = missing['wind_speed'].ffill().bfill()
missing['precipitation'] = missing['precipitation'].ffill().bfill()

final = pd.concat([climate, missing], ignore_index=True)
final = final.sort_values(['bird_name', 'date_time']).reset_index(drop=True)
final.to_csv('bird_migration_with_climate.csv', index=False)

print('Done. Final shape:', final.shape)
print('Nulls:')
print(final[['temperature','wind_speed','precipitation']].isnull().sum())

Missing rows: 419
Bird: ['Sanne']
Done. Final shape: (61920, 14)
Nulls:
temperature      0
wind_speed       0
precipitation    0
dtype: int64


In [ ]:
import requests
import time
import pandas as pd
import numpy as np

checkpoint = pd.read_csv('climate_checkpoint.csv')

def fetch_climate(lat, lon, date_str, hour):
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={round(lat,4)}&longitude={round(lon,4)}"
        f"&start_date={date_str}&end_date={date_str}"
        f"&hourly=temperature_2m,wind_speed_10m,precipitation"
        f"&timezone=UTC"
    )
    for attempt in range(3):
        try:
            r      = requests.get(url, timeout=15)
            d      = r.json()
            hourly = d.get('hourly', {})
            temp   = hourly.get('temperature_2m',  [np.nan]*24)[hour]
            wind   = hourly.get('wind_speed_10m',  [np.nan]*24)[hour]
            precip = hourly.get('precipitation',   [np.nan]*24)[hour]
            if temp is not None:
                return temp, wind, precip
        except:
            time.sleep(2)
    return np.nan, np.nan, np.nan

null_indices = checkpoint[checkpoint['temperature'].isnull()].index
print(f'Re-fetching {len(null_indices)} null rows...')

for count, idx in enumerate(null_indices):
    row = checkpoint.iloc[idx]
    t, w, p = fetch_climate(
        row['latitude'], row['longitude'],
        pd.to_datetime(row['date_time']).strftime('%Y-%m-%d'),
        int(pd.to_datetime(row['date_time']).hour)
    )
    checkpoint.at[idx, 'temperature']   = t
    checkpoint.at[idx, 'wind_speed']    = w
    checkpoint.at[idx, 'precipitation'] = p
    
    if count % 100 == 0:
        print(f'Fixed {count}/{len(null_indices)} null rows...')
    time.sleep(0.1)

checkpoint.to_csv('climate_checkpoint.csv', index=False)
print('Done. Nulls remaining:', checkpoint['temperature'].isnull().sum())

In [5]:
import pandas as pd

checkpoint = pd.read_csv('climate_checkpoint.csv')
print('Total rows:', len(checkpoint))
print('Null check:')
print(checkpoint[['temperature','wind_speed','precipitation']].isnull().sum())
print()
print('Sample of previously null rows now filled:')
print(checkpoint.loc[[1124, 1129, 2328, 2329, 2330], 
      ['latitude','longitude','temperature','wind_speed','precipitation']])

Total rows: 25501
Null check:
temperature      0
wind_speed       0
precipitation    0
dtype: int64

Sample of previously null rows now filled:
       latitude  longitude  temperature  wind_speed  precipitation
1124  50.050786   2.304363         21.0        13.7            0.0
1129  50.080523   2.353962         21.5        12.4            0.0
2328  50.191417   2.732767         11.7         8.8            0.0
2329  50.191447   2.733154         11.7         8.8            0.0
2330  50.191616   2.733243         11.3         8.6            0.0


In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv('bird_migration_with_climate.csv')
df['date_time'] = pd.to_datetime(df['date_time'], utc=True)
df = df.sort_values(['bird_name', 'date_time']).reset_index(drop=True)

print('df reloaded. Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Nulls:', df.isnull().sum().sum())

df reloaded. Shape: (61920, 14)
Columns: ['altitude', 'date_time', 'device_info_serial', 'direction', 'latitude', 'longitude', 'speed_2d', 'bird_name', 'unix_timestamp', 'day_of_year', 'hour', 'temperature', 'wind_speed', 'precipitation']
Nulls: 6


## 6. Feature Scaling (MinMaxScaler)

Each bird scaled independently — migration routes differ in geographic range.  
Scalers saved to `scalers.pkl` for inverse transform post-processing.

In [14]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pandas as pd

GRU_FEATURES   = ['latitude', 'longitude', 'unix_timestamp', 'day_of_year', 'hour']
TARGET_INDICES = [0, 1]
SEQ_LENGTH     = 10

scaled_data = {}
scalers     = {}

for bird in df['bird_name'].unique():
    bird_df = df[df['bird_name'] == bird][GRU_FEATURES].values

    scaler      = MinMaxScaler(feature_range=(0, 1))
    bird_scaled = scaler.fit_transform(bird_df)

    scaled_data[bird] = bird_scaled
    scalers[bird]     = scaler

    print(f'{bird}: raw shape {bird_df.shape} → scaled min={bird_scaled.min():.3f}, max={bird_scaled.max():.3f}')

Eric: raw shape (19795, 5) → scaled min=0.000, max=1.000
Nico: raw shape (21121, 5) → scaled min=0.000, max=1.000
Sanne: raw shape (21004, 5) → scaled min=0.000, max=1.000


In [15]:
import pickle

with open('scalers.pkl', 'wb') as f:
    pickle.dump(scalers, f)

print('Scalers saved to scalers.pkl')

Scalers saved to scalers.pkl


## 7. Sequence Creation

Sliding window: 10 consecutive GPS readings → predict next latitude and longitude.  
Input shape: `(n, 10, 5)` — Output shape: `(n, 2)` — Train/test: 80/20, no shuffle.

In [16]:
def create_sequences(data, target_indices, seq_length=10):
    """
    Convert 2D scaled array into GRU-ready sequences.

    Args:
        data           : numpy array (n_rows x n_features) — scaled
        target_indices : list of column indices to predict, e.g. [0, 1]
        seq_length     : number of past steps to use (confirmed: 10)

    Returns:
        X : shape (n_samples, seq_length, n_features)  — GRU input
        y : shape (n_samples, len(target_indices))     — GRU target
    """
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i - seq_length : i, :])
        y.append(data[i, target_indices])
    return np.array(X), np.array(y)

print('Function defined.')

Function defined.


In [17]:
sequences = {}

for bird in df['bird_name'].unique():
    data = scaled_data[bird]

    X, y = create_sequences(data, TARGET_INDICES, SEQ_LENGTH)

    split     = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    sequences[bird] = {
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test
    }

    print(f'{bird}:')
    print(f'  X_train: {X_train.shape}   y_train: {y_train.shape}')
    print(f'  X_test:  {X_test.shape}    y_test:  {y_test.shape}')

Eric:
  X_train: (15828, 10, 5)   y_train: (15828, 2)
  X_test:  (3957, 10, 5)    y_test:  (3957, 2)
Nico:
  X_train: (16888, 10, 5)   y_train: (16888, 2)
  X_test:  (4223, 10, 5)    y_test:  (4223, 2)
Sanne:
  X_train: (16795, 10, 5)   y_train: (16795, 2)
  X_test:  (4199, 10, 5)    y_test:  (4199, 2)


## 8. Save Outputs

Two deliverables for teammate:
1. GRU inputs: `X_train`, `X_test`, `y_train`, `y_test` per bird + `scalers.pkl`
2. Climate CSV: `bird_migration_with_climate.csv` for Gemini AI climate insights

In [18]:
for bird in sequences:
    np.save(f'X_train_{bird}.npy', sequences[bird]['X_train'])
    np.save(f'X_test_{bird}.npy',  sequences[bird]['X_test'])
    np.save(f'y_train_{bird}.npy', sequences[bird]['y_train'])
    np.save(f'y_test_{bird}.npy',  sequences[bird]['y_test'])

print('Sequence arrays saved for all 3 birds.')

Sequence arrays saved for all 3 birds.


In [19]:
import os

expected_files = (
    ['scalers.pkl', 'bird_migration_with_climate.csv'] +
    [f'{prefix}_{bird}.npy'
     for bird in ['Eric', 'Nico', 'Sanne']
     for prefix in ['X_train', 'X_test', 'y_train', 'y_test']]
)

print('=== FILES READY FOR TEAMMATE ===')
for f in expected_files:
    status = '✅' if os.path.exists(f) else '❌ MISSING'
    print(f'  {status}  {f}')

=== FILES READY FOR TEAMMATE ===
  ✅  scalers.pkl
  ✅  bird_migration_with_climate.csv
  ✅  X_train_Eric.npy
  ✅  X_test_Eric.npy
  ✅  y_train_Eric.npy
  ✅  y_test_Eric.npy
  ✅  X_train_Nico.npy
  ✅  X_test_Nico.npy
  ✅  y_train_Nico.npy
  ✅  y_test_Nico.npy
  ✅  X_train_Sanne.npy
  ✅  X_test_Sanne.npy
  ✅  y_train_Sanne.npy
  ✅  y_test_Sanne.npy


In [20]:
print('=== SHAPE SUMMARY FOR TEAMMATE ===')
print(f'GRU features (in order): {GRU_FEATURES}')
print(f'Sequence length: {SEQ_LENGTH}')
print(f'Target indices: {TARGET_INDICES} → latitude, longitude')
print()
for bird in sequences:
    s = sequences[bird]
    print(f'{bird}:')
    print(f'  X_train: {s["X_train"].shape}   X_test: {s["X_test"].shape}')
    print(f'  y_train: {s["y_train"].shape}    y_test: {s["y_test"].shape}')

=== SHAPE SUMMARY FOR TEAMMATE ===
GRU features (in order): ['latitude', 'longitude', 'unix_timestamp', 'day_of_year', 'hour']
Sequence length: 10
Target indices: [0, 1] → latitude, longitude

Eric:
  X_train: (15828, 10, 5)   X_test: (3957, 10, 5)
  y_train: (15828, 2)    y_test: (3957, 2)
Nico:
  X_train: (16888, 10, 5)   X_test: (4223, 10, 5)
  y_train: (16888, 2)    y_test: (4223, 2)
Sanne:
  X_train: (16795, 10, 5)   X_test: (4199, 10, 5)
  y_train: (16795, 2)    y_test: (4199, 2)


## 9. Inverse Transform (Post-processing)

Converts GRU scaled lat/lon predictions back to real-world coordinates using saved scalers.  
Runs after teammate shares prediction files.

In [ ]:
def inverse_transform_predictions(pred_scaled, scaler, n_features, target_indices):
    """
    Convert scaled lat/lon predictions back to real coordinates.

    Args:
        pred_scaled    : array (n_samples, 2) — scaled lat/lon from GRU
        scaler         : fitted MinMaxScaler for this bird
        n_features     : 5 (total GRU features)
        target_indices : [0, 1]

    Returns:
        real_coords : array (n_samples, 2) — real lat/lon
    """
    dummy = np.zeros((len(pred_scaled), n_features))

    dummy[:, target_indices] = pred_scaled

    inv = scaler.inverse_transform(dummy)

    return inv[:, target_indices]

print('Inverse transform function defined. Waiting for teammate predictions.')